# CIP alerting system

This notebook compares actual CIP cycles from the log files with the object-specific thresholds in `CIP Pre Rinse Timer details.xlsx`.

A cycle is defined as a continuous block of records for one object. A gap of more than 30 minutes starts the next cycle. The log's `Time In Seconds` is treated as cumulative within each step, so the longest recorded value for each step is used and step times are then added together.


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('datasets')
LOG_FILES = sorted(DATA_DIR.glob('CIP_Log*.xlsx'))
REFERENCE_FILE = DATA_DIR / 'CIP Pre Rinse Timer details.xlsx'

# These are deliberately visible and editable.
CYCLE_GAP_MINUTES = 30
DURATION_TOLERANCE = 0.10  # 10% over the reference duration
FLOW_TOLERANCE = 0.10      # 10% below the reference flow target

print(f'{len(LOG_FILES)} log files found')
print(f'Reference workbook: {REFERENCE_FILE.name}')


4 log files found
Reference workbook: CIP Pre Rinse Timer details.xlsx


In [3]:
REQUIRED = [
    'Date Time', 'Object', 'Recipe', 'Activity', 'Step',
    'Time In Seconds', 'Temp SP', 'Return Temp', 'Supply Flow',
    'Return Flow', 'Min Return Flow SP', 'Return Conducivity',
    'Caustic SP', 'Acid SP', 'Final Rinse Cond. SP', 'Flip'
]

def line_from_report(path):
    raw = pd.read_excel(path, header=None, nrows=10)
    text = ' '.join(raw.astype(str).fillna('').stack())
    match = re.search(r'Line\s*:\s*([^ ]+)', text, flags=re.I)
    return match.group(1) if match else path.stem

def read_log(path):
    df = pd.read_excel(path, sheet_name=0, header=13)
    df = df[REQUIRED].copy()
    df['Date Time'] = pd.to_datetime(df['Date Time'], errors='coerce')
    df = df[df['Date Time'].notna()].copy()
    for col in REQUIRED[5:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['Line'] = line_from_report(path)
    df['Source File'] = path.name
    return df

logs = pd.concat([read_log(path) for path in LOG_FILES], ignore_index=True)
logs = logs.sort_values(['Line', 'Object', 'Date Time']).reset_index(drop=True)
print(f'{len(logs):,} timestamped log rows loaded')
logs[['Line', 'Object', 'Activity', 'Step']].head()


38,943 timestamped log rows loaded


,Line,Object,Activity,Step
0,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
1,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
2,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill Circuit
3,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation
4,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation


In [4]:
summary = pd.read_excel(REFERENCE_FILE, sheet_name='Summary')
process = pd.read_excel(REFERENCE_FILE, sheet_name='Process CIP Water Consumption')

def object_key(value):
    # Uses the equipment code shared by the reference and log names, e.g. C02C01.
    match = re.search(r'C\d{2}[A-Z]\d{2}', str(value).upper())
    return match.group(0) if match else str(value).strip().upper()

summary['Object Key'] = summary['Object'].map(object_key)
process['Object Key'] = process['Object'].map(object_key)
logs['Object Key'] = logs['Object'].map(object_key)

final_rinse_seconds = [
    'Final Rinse Fill to Tank (sec)',
    'Final Rinse Fill Circuit (Sec)',
    'Final Rinse Front Mix Zone (Sec)',
    'Final Rinse Circulation (Sec)',
      ]

thresholds = summary.merge(
    process[['Object Key', 'Suppy Flow (L/Hr)', 'Return Flow (L/Hr)', *final_rinse_seconds]],
    on='Object Key',
    how='left'
)
thresholds['Final Rinse Total Time (min)'] = thresholds[final_rinse_seconds].sum(axis=1) / 60
thresholds = thresholds.rename(columns={
    'New Proposed Pre Rinse total time(Min)': 'Pre Rinse Threshold (min)',
    'Caustic- Total Time': 'Caustic Threshold (min)',
    'Acid Total Time': 'Acid Threshold (min)',
    'Suppy Flow (L/Hr)': 'Supply Flow Threshold (L/hr)',
    'Return Flow (L/Hr)': 'Return Flow Threshold (L/hr)'
})

threshold_view = thresholds[[
    'Object', 'Object Key', 'Pre Rinse Threshold (min)',
    'Caustic Threshold (min)', 'Acid Threshold (min)',
    'Final Rinse Total Time (min)', 'Supply Flow Threshold (L/hr)',
    'Return Flow Threshold (L/hr)'
]].copy()
threshold_view.head()


,Object,Object Key,Pre Rinse Threshold (min),Caustic Threshold (min),Acid Threshold (min),Final Rinse Total Time (min),Supply Flow Threshold (L/hr),Return Flow Threshold (L/hr)
0,M11W01_03_C02C01_PMT-1,C02C01,7.916667,21.666667,17.666667,8.666667,14000,12600
1,M11W01_04_C02C02_PMT-2,C02C02,7.916667,21.666667,17.666667,8.666667,14000,12600
2,M11W01_05_C02C03_PMT-3,C02C03,7.916667,21.666667,17.666667,8.333333,14000,12600
3,M11W01_06_C02C04_PMT-4,C02C04,7.916667,21.666667,17.666667,8.666667,14000,12600
4,M11W01_07_C02W01_PMT-RECIRCULATION,C02W01,5.033333,10.016667,11.016667,6.516667,16000,15000


### Final-rinse calculation

The final-rinse total is calculated as:

`Fill to Tank + Fill Circuit + Front Mix Zone + Circulation`

All four reference fields are in seconds and represent sequential final-rinse steps, so adding them and dividing by 60 is the correct first calculation. If the PLC runs any of these steps in parallel, that assumption should be changed after confirming it with the process owner.


In [5]:
logs = logs.sort_values(['Line', 'Object', 'Date Time']).copy()
new_cycle = (logs.groupby(['Line', 'Object'])['Date Time']
             .diff().gt(pd.Timedelta(minutes=CYCLE_GAP_MINUTES)))
logs['Cycle Number'] = (new_cycle.fillna(True)
                       .groupby([logs['Line'], logs['Object']])
                       .cumsum().astype(int))

cycle_keys = ['Line', 'Object', 'Object Key', 'Cycle Number']
cycle_info = (logs.groupby(cycle_keys, as_index=False)
              .agg({'Date Time': ['min', 'max', 'size'],
                    'Recipe': lambda x: ', '.join(sorted(set(x.dropna().astype(str))))}))
cycle_info.columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Record Count', 'Recipes']

step_times = (logs.groupby(cycle_keys + ['Activity', 'Step'], as_index=False)
              ['Time In Seconds'].max()
              .rename(columns={'Time In Seconds': 'Step Time (sec)'}))
step_times['Step Time (min)'] = step_times['Step Time (sec)'] / 60

activity_times = (step_times.groupby(cycle_keys + ['Activity'], as_index=False)
                  ['Step Time (min)'].sum()
                  .pivot(index=cycle_keys, columns='Activity', values='Step Time (min)')
                  .reset_index()
                  .rename_axis(None, axis=1))

flow_times = (logs.groupby(cycle_keys, as_index=False)
              .agg({'Supply Flow': 'median', 'Return Flow': ['median', 'min']}))
flow_times.columns = cycle_keys + ['Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)', 'Minimum Return Flow Seen (L/hr)']

cycle_results = (cycle_info.merge(activity_times, on=cycle_keys, how='left')
                .merge(flow_times, on=cycle_keys, how='left'))
cycle_results.head()


,Line,Object,Object Key,Cycle Number,Cycle Start,Cycle End,Record Count,Recipes,Acid,Caustic,Final Drain,Final Rinse,Mid Rinse,Pre Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Minimum Return Flow Seen (L/hr)
0,M01W01,C01C01-RMT Tank-1,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,37,Caustic,NaN,50.600000,36.150000,96.450000,NaN,8.450000,12784.38,20220.0,-1045.0
1,M01W01,C01C01-RMT Tank-1,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,54,Caustic,NaN,65.283333,53.083333,143.233333,NaN,8.216667,11404.69,267.5,-50.0
2,M01W01,C01C01-RMT Tank-1,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,81,Caustic Acid,185.15,77.533333,NaN,222.150000,152.15,12.566667,12403.13,3115.0,-1045.0
3,M01W01,C01C01-RMT Tank-1,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,69,Caustic,NaN,78.150000,NaN,315.900000,NaN,7.350000,862.50,-45.0,-1045.0
4,M01W01,C01C01-RMT Tank-1,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,61,Caustic,NaN,66.750000,NaN,168.750000,NaN,10.333333,281.25,-45.0,-560.0


In [6]:
actual_to_threshold = {
    'Pre Rinse': ('Pre Rinse', 'Pre Rinse Threshold (min)'),
    'Caustic': ('Caustic', 'Caustic Threshold (min)'),
    'Acid': ('Acid', 'Acid Threshold (min)'),
    'Final Rinse': ('Final Rinse', 'Final Rinse Total Time (min)')
}

threshold_lookup = thresholds.drop_duplicates('Object Key').set_index('Object Key')

for phase, (actual_col, threshold_col) in actual_to_threshold.items():
    cycle_results[f'{phase} Threshold (min)'] = cycle_results['Object Key'].map(
        threshold_lookup[threshold_col])
    cycle_results[f'{phase} Breach'] = (
        cycle_results[actual_col].notna()
        & cycle_results[f'{phase} Threshold (min)'].notna()
        & (cycle_results[actual_col] > cycle_results[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE))
    )

cycle_results['Supply Flow Threshold (L/hr)'] = cycle_results['Object Key'].map(
    threshold_lookup['Supply Flow Threshold (L/hr)'])
cycle_results['Return Flow Threshold (L/hr)'] = cycle_results['Object Key'].map(
    threshold_lookup['Return Flow Threshold (L/hr)'])
cycle_results['Supply Flow Breach'] = (
    cycle_results['Supply Flow Actual (L/hr)'].notna()
    & cycle_results['Supply Flow Threshold (L/hr)'].notna()
    & (cycle_results['Supply Flow Actual (L/hr)'] < cycle_results['Supply Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)
cycle_results['Return Flow Breach'] = (
    cycle_results['Return Flow Actual (L/hr)'].notna()
    & cycle_results['Return Flow Threshold (L/hr)'].notna()
    & (cycle_results['Return Flow Actual (L/hr)'] < cycle_results['Return Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)

breach_cols = [c for c in cycle_results.columns if c.endswith('Breach')]
cycle_results['Alert Count'] = cycle_results[breach_cols].sum(axis=1)
cycle_results['Status'] = np.where(cycle_results['Alert Count'] > 0, 'ALERT', 'OK')

cycle_summary_columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Recipes',
    'Pre Rinse', 'Caustic', 'Acid', 'Final Rinse',
    'Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)',
    'Alert Count', 'Status']
cycle_results[cycle_summary_columns].head(20)


,Line,Object,Object Key,Cycle Number,Cycle Start,Cycle End,Recipes,Pre Rinse,Caustic,Acid,Final Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Alert Count,Status
0,M01W01,C01C01-RMT Tank-1,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,Caustic,8.450000,50.600000,NaN,96.450000,12784.3800,20220.0,0,OK
1,M01W01,C01C01-RMT Tank-1,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,Caustic,8.216667,65.283333,NaN,143.233333,11404.6900,267.5,0,OK
2,M01W01,C01C01-RMT Tank-1,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,Caustic Acid,12.566667,77.533333,185.150000,222.150000,12403.1300,3115.0,0,OK
3,M01W01,C01C01-RMT Tank-1,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,Caustic,7.350000,78.150000,NaN,315.900000,862.5000,-45.0,0,OK
4,M01W01,C01C01-RMT Tank-1,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,Caustic,10.333333,66.750000,NaN,168.750000,281.2500,-45.0,0,OK
5,M01W01,C01C01-RMT Tank-1,C01C01,5,2026-07-08 07:32:06.703,2026-07-08 08:10:06.653,Caustic,11.583333,59.550000,NaN,104.200000,12643.7500,11405.0,0,OK
6,M01W01,C01C01-RMT Tank-1,C01C01,6,2026-07-09 14:57:06.703,2026-07-09 15:40:06.703,Caustic,11.300000,77.650000,NaN,118.000000,12745.3100,18422.5,0,OK
7,M01W01,C01C01-RMT Tank-1,C01C01,7,2026-07-10 21:49:06.653,2026-07-10 22:52:06.703,"Caustic, Caustic Acid",8.483333,44.483333,141.450000,173.466667,12748.4400,15887.5,0,OK
8,M01W01,C01C01-RMT Tank-1,C01C01,8,2026-07-12 09:04:06.653,2026-07-12 09:48:06.703,Caustic,7.150000,57.816667,NaN,123.150000,11703.1300,-40.0,0,OK
9,M01W01,C01C01-RMT Tank-1,C01C01,9,2026-07-12 17:47:06.703,2026-07-12 17:55:06.653,"Caustic, Final rinse",0.083333,NaN,NaN,8.800000,8087.5000,-40.0,0,OK


In [7]:
alert_rows = []
for _, row in cycle_results.iterrows():
    base = row[cycle_keys + ['Cycle Start', 'Cycle End']].to_dict()
    for phase, (actual_col, threshold_col) in actual_to_threshold.items():
        if bool(row[f'{phase} Breach']):
            alert_rows.append({**base, 'Alert Type': f'{phase} duration',
                               'Actual Value': row[actual_col], 'Threshold': row[f'{phase} Threshold (min)'],
                               'Unit': 'minutes'})
    for signal, actual_col, threshold_col in [
        ('Supply flow', 'Supply Flow Actual (L/hr)', 'Supply Flow Threshold (L/hr)'),
        ('Return flow', 'Return Flow Actual (L/hr)', 'Return Flow Threshold (L/hr)')
    ]:
        if bool(row[f'{signal.title()} Breach']):
            alert_rows.append({**base, 'Alert Type': signal,
                               'Actual Value': row[actual_col], 'Threshold': row[threshold_col],
                               'Unit': 'L/hr'})

alerts = pd.DataFrame(alert_rows)
if alerts.empty:
    alerts = pd.DataFrame(columns=cycle_keys + ['Cycle Start', 'Cycle End', 'Alert Type', 'Actual Value', 'Threshold', 'Unit'])

print(f'{len(cycle_results):,} cycles identified')
print(f'{len(alerts):,} threshold breaches found')
alerts.head(50)


699 cycles identified
1,460 threshold breaches found


,Line,Object,Object Key,Cycle Number,Cycle Start,Cycle End,Alert Type,Actual Value,Threshold,Unit
0,M11W01,C01W01-Milk Past-1,C01W01,0,2026-07-01 16:27:06.703,2026-07-01 17:14:06.703,Caustic duration,59.333333,33.5,minutes
1,M11W01,C01W01-Milk Past-1,C01W01,0,2026-07-01 16:27:06.703,2026-07-01 17:14:06.703,Final Rinse duration,205.866667,16.0,minutes
2,M11W01,C01W01-Milk Past-1,C01W01,1,2026-07-02 07:54:06.653,2026-07-02 08:53:06.653,Pre Rinse duration,31.800000,14.5,minutes
3,M11W01,C01W01-Milk Past-1,C01W01,1,2026-07-02 07:54:06.653,2026-07-02 08:53:06.653,Caustic duration,130.750000,33.5,minutes
4,M11W01,C01W01-Milk Past-1,C01W01,1,2026-07-02 07:54:06.653,2026-07-02 08:53:06.653,Final Rinse duration,217.000000,16.0,minutes
5,M11W01,C01W01-Milk Past-1,C01W01,2,2026-07-02 20:13:06.703,2026-07-02 20:58:06.703,Pre Rinse duration,27.183333,14.5,minutes
6,M11W01,C01W01-Milk Past-1,C01W01,2,2026-07-02 20:13:06.703,2026-07-02 20:58:06.703,Caustic duration,57.300000,33.5,minutes
7,M11W01,C01W01-Milk Past-1,C01W01,2,2026-07-02 20:13:06.703,2026-07-02 20:58:06.703,Final Rinse duration,161.733333,16.0,minutes
8,M11W01,C01W01-Milk Past-1,C01W01,3,2026-07-03 06:40:06.703,2026-07-03 07:32:06.653,Pre Rinse duration,22.300000,14.5,minutes
9,M11W01,C01W01-Milk Past-1,C01W01,3,2026-07-03 06:40:06.703,2026-07-03 07:32:06.653,Caustic duration,87.316667,33.5,minutes


In [8]:
unmatched = (cycle_results.loc[~cycle_results['Object Key'].isin(set(thresholds['Object Key'])),
                             ['Line', 'Object', 'Object Key']]
             .drop_duplicates()
             .sort_values(['Line', 'Object']))

cycle_counts = (cycle_results.groupby(['Line', 'Object'], as_index=False)
                .agg(Cycles=('Cycle Number', 'nunique'))
                .sort_values(['Line', 'Object']))
print('Cycles per object:')
cycle_counts.head(50)

print('Objects without a matching reference threshold:')
unmatched


Cycles per object:
Objects without a matching reference threshold:


,Line,Object,Object Key
0,M01W01,C01C01-RMT Tank-1,C01C01
21,M01W01,C01C02-RMT Tank-2,C01C02
44,M01W01,C01C03-RMT Tank-3,C01C03
71,M01W01,C01W03-RMT To Reception line,C01W03
92,M01W01,R01W01-Milk Reception Line,R01W01-MILK RECEPTION LINE
116,M01W02,M01W02_CC01 - Raw Milk Tanker,M01W02_CC01 - RAW MILK TANKER
421,M11W02,C01W02-Milk Past-2,C01W02


## How to use this notebook

Run the cells from top to bottom. The main outputs are `cycle_results`, which contains one row per object cycle, and `alerts`, which contains one row per breach.

The current flow comparison uses the median flow observed during a cycle. That avoids triggering an alert from one bad sensor reading. The duration and flow tolerances are working defaults, not confirmed plant rules. Confirm them with the CIP owner before treating the alerts as operational alarms.
